# KKBOX Tab 2 Predictive Training Notebook

Notebook nay duoc viet cho Kaggle va chi dung feature store batch da rerun lam dau vao.

Scope canonical cua notebook:
- train churn classifier cho expiring cohort;
- xuat `churn_probability` va `expected_revenue_at_risk_30d`;
- xuat artifact de Tab 2 doc lai tren UI.

Notebook nay khong build lai feature tu raw transaction / raw user_logs, va khong overclaim Cox / CLTV neu chua co pipeline rieng.


In [ ]:
from pathlib import Path
import json

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

PROJECT_DIR = Path.cwd()
FEATURE_STORE_ROOT_HINT = None
OUTPUT_DIR = PROJECT_DIR / 'artifacts_tab2_predictive'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_MONTHS = [201701, 201702]
VALID_MONTH = 201703
SCORE_MONTH = 201704
RANDOM_SEED = 3228

print(f'PROJECT_DIR = {PROJECT_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')


In [ ]:
def resolve_feature_store_dir(root_hint=None) -> Path:
    candidates = []
    if root_hint is not None:
        root_hint = Path(root_hint)
        candidates.extend([root_hint, root_hint / 'feature_store'])

    candidates.extend([
        Path('/kaggle/input/kkbox-feature-store'),
        Path('/kaggle/input/kkbox-feature-store') / 'feature_store',
        Path('/kaggle/input/kkbox-churn-feature-store'),
        Path('/kaggle/input/kkbox-churn-feature-store') / 'feature_store',
        Path('/kaggle/input/kkbox-churn-output'),
        Path('/kaggle/input/kkbox-churn-output') / 'feature_store',
        PROJECT_DIR / 'artifacts' / 'feature_store',
        PROJECT_DIR / 'feature_store',
    ])

    required_files = [
        'train_features_bi_all.parquet',
        f'test_features_bi_{SCORE_MONTH}_full.parquet',
        'feature_columns.csv',
        'bi_dimension_columns.csv',
    ]

    for candidate in candidates:
        if not candidate.exists():
            continue
        if all((candidate / name).exists() for name in required_files):
            return candidate

    raise FileNotFoundError(
        'Khong tim thay feature store. Hay gan output cua train_churn_pipeline nhu mot Kaggle input dataset.'
    )


FEATURE_STORE_DIR = resolve_feature_store_dir(FEATURE_STORE_ROOT_HINT)
TRAIN_BI_PATH = FEATURE_STORE_DIR / 'train_features_bi_all.parquet'
TEST_BI_PATH = FEATURE_STORE_DIR / f'test_features_bi_{SCORE_MONTH}_full.parquet'
FEATURE_COLUMNS_PATH = FEATURE_STORE_DIR / 'feature_columns.csv'
BI_DIMENSIONS_PATH = FEATURE_STORE_DIR / 'bi_dimension_columns.csv'

print(f'FEATURE_STORE_DIR = {FEATURE_STORE_DIR}')
print(f'TRAIN_BI_PATH = {TRAIN_BI_PATH}')
print(f'TEST_BI_PATH = {TEST_BI_PATH}')


In [ ]:
train_df = pd.read_parquet(TRAIN_BI_PATH)
test_df = pd.read_parquet(TEST_BI_PATH)
feature_columns = pd.read_csv(FEATURE_COLUMNS_PATH)['feature'].tolist()
bi_dimensions = pd.read_csv(BI_DIMENSIONS_PATH)['dimension'].tolist()

required_train_cols = ['msno', 'target_month', 'is_churn', 'expected_renewal_amount']
required_test_cols = ['msno', 'target_month', 'expected_renewal_amount']

missing_train = [col for col in required_train_cols if col not in train_df.columns]
missing_test = [col for col in required_test_cols if col not in test_df.columns]
if missing_train:
    raise ValueError(f'Train BI parquet thieu cot bat buoc: {missing_train}')
if missing_test:
    raise ValueError(f'Test BI parquet thieu cot bat buoc: {missing_test}')

available_train_months = sorted(train_df['target_month'].astype('int32').unique().tolist())
available_test_months = sorted(test_df['target_month'].astype('int32').unique().tolist())
print('train_df shape:', train_df.shape)
print('test_df shape:', test_df.shape)
print('available_train_months:', available_train_months)
print('available_test_months:', available_test_months)

feature_exclude = {'msno', 'is_churn', 'transaction_date', 'expire_date'}
feature_cols = [
    col for col in feature_columns
    if col in train_df.columns and col in test_df.columns and col not in feature_exclude
]

if not feature_cols:
    raise ValueError('Khong co feature nao de train sau khi intersect feature_columns.csv voi train/test parquet.')

print('feature count:', len(feature_cols))
display(pd.Series(feature_cols, name='feature').head(30))
display(pd.Series(bi_dimensions, name='bi_dimension').head(20))


In [ ]:
def evaluate_predictions(y_true, y_prob):
    y_prob = np.clip(np.asarray(y_prob, dtype='float64'), 1e-15, 1 - 1e-15)
    return {
        'log_loss': float(log_loss(y_true, y_prob)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
        'positive_rate': float(np.mean(y_true)),
        'prediction_mean': float(np.mean(y_prob)),
    }


def make_feature_group_map(features):
    payment_features = {
        'payment_method_id', 'payment_plan_days', 'plan_list_price', 'actual_amount_paid', 'is_auto_renew',
        'discount', 'is_discount', 'amt_per_day', 'expected_renewal_amount', 'discount_ratio', 'payment_to_list_ratio',
    }
    churn_history_features = {
        'last_1_is_churn', 'last_2_is_churn', 'last_3_is_churn', 'last_4_is_churn', 'last_5_is_churn',
        'churn_rate', 'churn_count', 'transaction_count', 'historical_transaction_rows', 'historical_cancel_count',
        'historical_cancel_rate', 'historical_auto_renew_rate', 'weighted_recent_churn', 'recent_churn_events',
        'days_since_previous_transaction', 'churn_rate_x_transaction_count',
    }
    listening_features = {
        'count', 'num_25_sum', 'num_50_sum', 'num_75_sum', 'num_985_sum', 'num_100_sum', 'num_unq_sum',
        'total_secs_sum', 'listen_events_sum', 'skip_events_sum', 'skip_ratio', 'discovery_ratio', 'completion_ratio',
        'replay_ratio', 'days_since_last_listen', 'capped_log_share', 'secs_per_log', 'unique_per_log', 'avg_secs_per_unique',
        'secs_per_plan_day', 'uniques_per_plan_day', 'logs_per_plan_day', 'secs_per_paid_amount',
    }
    loyalty_features = {
        'city', 'gender', 'registered_via', 'age', 'has_valid_age', 'days_to_expire', 'membership_age_days',
        'tenure_months', 'remaining_plan_ratio', 'transaction_month', 'expire_month', 'transaction_day', 'expire_day',
        'registration_year', 'registration_month', 'registration_day', 'target_month',
    }
    segment_features = {
        'age_segment_code', 'price_segment_code', 'loyalty_segment_code', 'active_segment_code',
        'skip_segment_code', 'discovery_segment_code', 'renewal_segment_code', 'rfm_segment_code',
        'deal_hunter_flag', 'free_trial_flag', 'content_fatigue_flag', 'high_skip_flag', 'low_discovery_flag',
        'is_manual_renew', 'auto_renew_discount_interaction', 'rfm_total_score', 'rfm_recency_score',
        'rfm_frequency_score', 'rfm_monetary_score', 'weighted_completion_sum', 'weighted_completion_per_log',
    }

    group_map = {}
    for feature in features:
        if feature in payment_features:
            group_map[feature] = 'payment_value'
        elif feature in churn_history_features:
            group_map[feature] = 'churn_history'
        elif feature in listening_features:
            group_map[feature] = 'listening_behavior'
        elif feature in loyalty_features:
            group_map[feature] = 'loyalty_member'
        elif feature in segment_features:
            group_map[feature] = 'segment_flags'
        else:
            group_map[feature] = 'other'
    return group_map


def attach_tab2_outputs(df, raw_prob, final_prob, extra_dimension_cols=None):
    out = df[['msno', 'target_month']].copy()
    if 'is_churn' in df.columns:
        out['is_churn'] = df['is_churn'].astype('int8')
    safe_amount = df['expected_renewal_amount'].fillna(0).clip(lower=0).astype('float32')
    out['expected_renewal_amount'] = safe_amount
    out['churn_probability_raw'] = np.asarray(raw_prob, dtype='float32')
    out['churn_probability'] = np.asarray(final_prob, dtype='float32')
    out['risk_percentile'] = pd.Series(out['churn_probability']).rank(method='average', pct=True).astype('float32')
    out['risk_decile'] = np.ceil(out['risk_percentile'] * 10).clip(1, 10).astype('int8')
    out['risk_band'] = np.select(
        [
            out['churn_probability'] >= 0.80,
            out['churn_probability'] >= 0.60,
            out['churn_probability'] >= 0.40,
            out['churn_probability'] >= 0.20,
        ],
        ['Very High', 'High', 'Medium', 'Low'],
        default='Very Low',
    )
    out['expected_revenue_at_risk_30d'] = (out['churn_probability'] * safe_amount).astype('float32')
    out['expected_retained_revenue_30d'] = ((1 - out['churn_probability']) * safe_amount).astype('float32')
    if extra_dimension_cols:
        keep_cols = [col for col in extra_dimension_cols if col in df.columns]
        out = pd.concat([out.reset_index(drop=True), df[keep_cols].reset_index(drop=True)], axis=1)
    return out


feature_group_map = make_feature_group_map(feature_cols)
extra_dimension_cols = [
    'price_segment', 'renewal_segment', 'loyalty_segment', 'active_segment',
    'skip_segment', 'discovery_segment', 'bi_segment_name', 'gender_profile'
]


In [ ]:
train_fit_df = train_df[train_df['target_month'].isin(TRAIN_MONTHS)].reset_index(drop=True)
valid_df = train_df[train_df['target_month'] == VALID_MONTH].reset_index(drop=True)
full_train_df = train_df[train_df['target_month'].isin(TRAIN_MONTHS + [VALID_MONTH])].reset_index(drop=True)
score_df = test_df[test_df['target_month'] == SCORE_MONTH].reset_index(drop=True)

if train_fit_df.empty or valid_df.empty or full_train_df.empty or score_df.empty:
    raise ValueError(
        f'Time split rong. train={len(train_fit_df)}, valid={len(valid_df)}, full={len(full_train_df)}, score={len(score_df)}'
    )

X_train = train_fit_df[feature_cols].fillna(-1)
y_train = train_fit_df['is_churn'].astype('int8')
X_valid = valid_df[feature_cols].fillna(-1)
y_valid = valid_df['is_churn'].astype('int8')
X_full = full_train_df[feature_cols].fillna(-1)
y_full = full_train_df['is_churn'].astype('int8')
X_score = score_df[feature_cols].fillna(-1)

pos = float(y_train.sum())
neg = float(len(y_train) - y_train.sum())
scale_pos_weight = max(neg / max(pos, 1.0), 1.0)

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.03,
    'num_leaves': 96,
    'min_data_in_leaf': 120,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'lambda_l1': 0.5,
    'lambda_l2': 2.0,
    'scale_pos_weight': scale_pos_weight,
    'verbosity': -1,
    'seed': RANDOM_SEED,
}

train_set = lgb.Dataset(X_train, label=y_train)
valid_set = lgb.Dataset(X_valid, label=y_valid, reference=train_set)

model_valid = lgb.train(
    params=lgb_params,
    train_set=train_set,
    num_boost_round=2000,
    valid_sets=[train_set, valid_set],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100),
    ],
)

valid_raw_pred = np.clip(model_valid.predict(X_valid, num_iteration=model_valid.best_iteration), 1e-15, 1 - 1e-15)
raw_metrics = evaluate_predictions(y_valid, valid_raw_pred)

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(valid_raw_pred, y_valid)
valid_calibrated_pred = np.clip(calibrator.transform(valid_raw_pred), 1e-15, 1 - 1e-15)
calibrated_metrics = evaluate_predictions(y_valid, valid_calibrated_pred)

use_calibrated = calibrated_metrics['log_loss'] < raw_metrics['log_loss']
final_valid_pred = valid_calibrated_pred if use_calibrated else valid_raw_pred
selected_metrics = calibrated_metrics if use_calibrated else raw_metrics

print('best_iteration:', model_valid.best_iteration)
print('use_calibrated_output:', use_calibrated)
display(pd.DataFrame([
    {'variant': 'raw', **raw_metrics},
    {'variant': 'calibrated', **calibrated_metrics},
]).sort_values('log_loss'))


In [ ]:
model_full = lgb.train(
    params=lgb_params,
    train_set=lgb.Dataset(X_full, label=y_full),
    num_boost_round=int(model_valid.best_iteration),
    valid_sets=[lgb.Dataset(X_full, label=y_full)],
    valid_names=['full_train'],
    callbacks=[lgb.log_evaluation(period=200)],
)

score_raw_pred = np.clip(model_full.predict(X_score, num_iteration=int(model_valid.best_iteration)), 1e-15, 1 - 1e-15)
score_final_pred = np.clip(calibrator.transform(score_raw_pred), 1e-15, 1 - 1e-15) if use_calibrated else score_raw_pred

valid_scored = attach_tab2_outputs(
    df=valid_df,
    raw_prob=valid_raw_pred,
    final_prob=final_valid_pred,
    extra_dimension_cols=extra_dimension_cols,
)
score_scored = attach_tab2_outputs(
    df=score_df,
    raw_prob=score_raw_pred,
    final_prob=score_final_pred,
    extra_dimension_cols=extra_dimension_cols,
)

segment_group_col = 'bi_segment_name'
if segment_group_col not in score_scored.columns:
    score_scored[segment_group_col] = 'Unknown'

segment_risk_summary = (
    score_scored.groupby(['target_month', segment_group_col], as_index=False)
    .agg(
        users=('msno', 'nunique'),
        avg_churn_probability=('churn_probability', 'mean'),
        total_expected_revenue_at_risk_30d=('expected_revenue_at_risk_30d', 'sum'),
        total_expected_renewal_amount=('expected_renewal_amount', 'sum'),
    )
    .sort_values(['total_expected_revenue_at_risk_30d', 'avg_churn_probability'], ascending=[False, False])
    .reset_index(drop=True)
)

display(valid_scored.head())
display(segment_risk_summary.head(20))


In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance_gain': model_full.feature_importance(importance_type='gain'),
    'importance_split': model_full.feature_importance(importance_type='split'),
})
feature_importance['feature_group'] = feature_importance['feature'].map(feature_group_map)
feature_importance = feature_importance.sort_values('importance_gain', ascending=False).reset_index(drop=True)

feature_group_importance = (
    feature_importance.groupby('feature_group', as_index=False)
    .agg(
        importance_gain=('importance_gain', 'sum'),
        importance_split=('importance_split', 'sum'),
        feature_count=('feature', 'count'),
    )
    .sort_values('importance_gain', ascending=False)
    .reset_index(drop=True)
)

validation_metrics_payload = {
    'train_months': TRAIN_MONTHS,
    'valid_month': VALID_MONTH,
    'score_month': SCORE_MONTH,
    'best_iteration': int(model_valid.best_iteration),
    'scale_pos_weight': float(scale_pos_weight),
    'use_calibrated_output': bool(use_calibrated),
    'raw_metrics': raw_metrics,
    'selected_metrics': selected_metrics,
    'feature_count': len(feature_cols),
}

model_summary_payload = {
    'artifacts_dir': str(OUTPUT_DIR),
    'feature_store_dir': str(FEATURE_STORE_DIR),
    'feature_count': len(feature_cols),
    'scored_validation_rows': int(len(valid_scored)),
    'scored_test_rows': int(len(score_scored)),
    'top_feature_groups': feature_group_importance.to_dict(orient='records'),
}

metrics_path = OUTPUT_DIR / 'tab2_validation_metrics.json'
feature_cols_path = OUTPUT_DIR / 'tab2_feature_columns_used.csv'
feature_importance_path = OUTPUT_DIR / 'tab2_feature_importance_lightgbm.csv'
feature_group_importance_path = OUTPUT_DIR / 'tab2_feature_group_importance.csv'
valid_scored_path = OUTPUT_DIR / f'tab2_valid_scored_{VALID_MONTH}.parquet'
score_scored_path = OUTPUT_DIR / f'tab2_test_scored_{SCORE_MONTH}.parquet'
segment_summary_path = OUTPUT_DIR / f'tab2_segment_risk_summary_{SCORE_MONTH}.parquet'
model_summary_path = OUTPUT_DIR / 'tab2_model_summary.json'
model_path = OUTPUT_DIR / 'tab2_lightgbm_model.txt'

metrics_path.write_text(json.dumps(validation_metrics_payload, indent=2, ensure_ascii=True))
feature_cols_path.write_text('feature\n' + '\n'.join(feature_cols) + '\n')
feature_importance.to_csv(feature_importance_path, index=False)
feature_group_importance.to_csv(feature_group_importance_path, index=False)
valid_scored.to_parquet(valid_scored_path, index=False)
score_scored.to_parquet(score_scored_path, index=False)
segment_risk_summary.to_parquet(segment_summary_path, index=False)
model_summary_path.write_text(json.dumps(model_summary_payload, indent=2, ensure_ascii=True))
model_full.save_model(str(model_path))

print('Saved artifacts:')
print(f'- metrics: {metrics_path}')
print(f'- feature cols: {feature_cols_path}')
print(f'- feature importance: {feature_importance_path}')
print(f'- feature group importance: {feature_group_importance_path}')
print(f'- valid scored: {valid_scored_path}')
print(f'- test scored: {score_scored_path}')
print(f'- segment risk summary: {segment_summary_path}')
print(f'- model summary: {model_summary_path}')
print(f'- model: {model_path}')

display(feature_importance.head(30))
display(feature_group_importance)
display(score_scored[['msno', 'target_month', 'churn_probability', 'expected_revenue_at_risk_30d', 'risk_band']].head(20))
